# Notebook B: extended-version experiments

Runs the leader's tested scripts, unchanged:
1. **Learning-rate check** (about 35 min): extended generation 0 at three learning rates, chosen on the validation set.
2. **Main runs** (about 3 h): `run_extended_conditions.sh`: `gen0_p50`, argmax (3 generations), T1 (10 generations).

Results go to **Drive → MyDrive → nlp_p1 → collapse_ext**. **Runtime: T4 GPU.** Keep the tab open. If Colab disconnects, run Setup 1–4 again, then rerun the cell that was running; finished runs are skipped.

## Setup 1: connect Google Drive
A pop-up asks for permission: pick your account → **Allow**. The last lines must say `collapse found: True`. If it says False, the shortcut from guide Part 3 is missing or has a different name.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
print('collapse found:', os.path.exists('/content/drive/MyDrive/collapse/gen0_pool/config.json'))
os.makedirs('/content/drive/MyDrive/nlp_p1/results', exist_ok=True)
os.makedirs('/content/drive/MyDrive/nlp_p1/collapse_ext', exist_ok=True)
!nvidia-smi --query-gpu=name --format=csv,noheader || echo "no GPU (fine for analysis, slower)"


## Setup 2: Python 3.10 with the project's versions (2–4 min)
The last line should show `0.4.26` and a device list.

In [ ]:
!pip -q install uv
!uv venv /content/py310 --python 3.10 -q --allow-existing
!uv pip install -q --python /content/py310/bin/python "jax[cuda12]==0.4.26" "jaxlib==0.4.26" "equinox==0.11.10" "optax==0.2.2" "nptyping==2.5.0" "numpy==1.26.4" "scipy==1.15.3" "h5py==3.16.0" "matplotlib==3.10.9" "tqdm==4.70.1"
!XLA_PYTHON_CLIENT_PREALLOCATE=false /content/py310/bin/python -c "import jax; print(jax.__version__, jax.devices())"


## Setup 3: the code (`extended` branch)


In [ ]:
BRANCH = 'extended'
%cd /content
!rm -rf icl-dynamics && git clone -q -b $BRANCH https://github.com/Bigjay101/icl-dynamics.git
%cd /content/icl-dynamics
!git log --oneline -1

## Setup 4: links and settings (run this every time you reconnect)
This makes `collapse/`, `collapse_ext/` and `results/` inside the code folder point to your Drive, so every command in the guide works unchanged and every output lands in Drive.

In [ ]:
%cd /content/icl-dynamics
!ln -sfn /content/drive/MyDrive/collapse collapse
!ln -sfn /content/drive/MyDrive/nlp_p1/collapse_ext collapse_ext
!rm -rf results && ln -sfn /content/drive/MyDrive/nlp_p1/results results
import os
os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'
os.environ['PY'] = '/content/py310/bin/python'
os.environ['TRAIN_ARGS'] = "--data_file omniglot_resnet18_randomized_order_s0.h5 --mixing_coeffs 1.0 --pt_burstiness 1 --train_context_len 2 --fs_relabel 5 --fs_relabel_split 8 2 0 --exemplar_split 1 4 0 --pe_names fsl_train fsl_val_rl fsl_train_valex fsl_test_class --pe_classes train train train test --pe_exemplars train train val train --pe_fs_relabel_scheme train val train train --pe_burstiness 1 1 1 1 --train_bs 32 --eval_iters 1000 --lr 0.00001 --d_model 64 --class_split 50 1473 100 --raw_name"
!ls -la | grep -E "collapse|results"


## Step A: learning-rate check (about 35 min)
Extended generation 0 (1M sequences, p_repeat 0.5, init seed 5) at **1e-5, 3e-5, 1e-4**.

In [ ]:
%%bash
cd /content/icl-dynamics
mkdir -p collapse_ext/lr_sweep
for LR in 0.00001 0.00003 0.0001; do
  RUN=lr_$LR
  if [ -f collapse_ext/lr_sweep/$RUN/checkpoints/00001000000.eqx ]; then echo "$RUN already done"; continue; fi
  rm -rf collapse_ext/lr_sweep/$RUN
  echo "=== training $RUN ($(date +%H:%M)) ==="
  $PY main.py $TRAIN_ARGS --extended --p_repeat 0.5 --lr $LR --train_iters 1000000 \
    --eval_every 5000 --ckpt_every 10000 --init_seed 5 --pool_size 1000000 \
    --base_folder collapse_ext/lr_sweep --run $RUN > collapse_ext/lr_sweep/$RUN.log 2>&1 || { echo "FAILED, end of log:"; tail -n 20 collapse_ext/lr_sweep/$RUN.log; exit 1; }
  grep "total training time" collapse_ext/lr_sweep/$RUN.log
done

## Step B: choose the learning rate (lowest validation `total_loss`)
The table is saved to `results/lr_sweep_extended.csv` for your method section.

In [ ]:
import h5py, numpy as np, os, glob, csv
rows = []
for run in sorted(glob.glob('collapse_ext/lr_sweep/lr_*/log.h5')):
    with h5py.File(run) as f:
        get = lambda k: float(np.nanmean(f['fsl_val_rl/' + k][-1])) if 'fsl_val_rl/' + k in f else float('nan')
        rows.append({'run': os.path.basename(os.path.dirname(run)), 'val_total_loss': get('total_loss'),
                     'val_query_loss': get('loss'), 'val_in_context_acc': get('in_context_acc'),
                     'val_next_label_in_context_acc': get('next_label_in_context_acc'),
                     'val_symbol_loss': get('symbol_loss'), 'val_symbol_use_context_prob': get('symbol_use_context_prob')})
for r in rows: print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in r.items()})
with open('results/lr_sweep_extended.csv', 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0])); w.writeheader(); w.writerows(rows)
best = min(rows, key=lambda r: r['val_total_loss'])
os.environ['LR'] = best['run'].replace('lr_', '')
print('\nCHOSEN LEARNING RATE:', os.environ['LR'], '| symbol-loss floor for p = 0.5 is ln 2 = 0.693')

## Step C: main extended runs (about 3 h)
If you reconnected after Step B, type the chosen learning rate in the first cell.

In [ ]:
import os
os.environ['LR'] = os.environ.get('LR', '0.00003')   # <- set by Step B; type it here if you reconnected
print('LR =', os.environ['LR'])

In [ ]:
%%bash
cd /content/icl-dynamics
LR=$LR BASE=collapse_ext PY=$PY bash run_extended_conditions.sh 2>&1 | tee -a collapse_ext/run_all.log

## Step D: check, and the leader's drift analysis

In [ ]:
%%bash
cd /content/icl-dynamics
for c in gen0_p50 argmax T1; do echo "== $c: $(ls collapse_ext/$c 2>/dev/null | tr '\n' ' ')"; done
$PY analyze_extended.py collapse_ext/T1 collapse_ext/gen0_p50 --out results/extended_drift 2>&1 | tail -n 20
$PY check_extended_run.py collapse_ext/gen0_p50 --full | tail -n 12

In [ ]:
from IPython.display import Image, display
display(Image('results/extended_drift/drift.png', width=900))

## Done
Next: Notebook A, Steps 5–6 (Q1 on the extended version).